# Example: Multi-Input Multi-Output (MIMO) Frequency Response Estimation

This example demonstrates MIMO system identification with `sid.freq_bt`.
Key differences from SISO: `response` is a 3-D array, `noise_spectrum` is a
spectral matrix, and `coherence` is not available.

Translated from `exampleMIMO.m`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lfilter
import sid

%matplotlib inline

## 2-output, 1-input system

Two independent channels driven by the same input:

- $G_1(z) = 1 / (1 - 0.5\,z^{-1})$
- $G_2(z) = 0.3 / (1 - 0.7\,z^{-1})$

In [ ]:
rng = np.random.default_rng(10)

N = 3000
u = rng.standard_normal(N)
y1 = lfilter([1],   [1, -0.5], u) + 0.1 * rng.standard_normal(N)
y2 = lfilter([0.3], [1, -0.7], u) + 0.1 * rng.standard_normal(N)
y = np.column_stack([y1, y2])   # (N x 2) output matrix

result = sid.freq_bt(y, u, window_size=30)

## Inspect MIMO result dimensions

`response` is `(nf, ny, nu)` = `(nf, 2, 1)` for this system.
`coherence` is `None` for MIMO.

In [ ]:
print(f'Response shape:       {result.response.shape}')
print(f'Noise spectrum shape: {result.noise_spectrum.shape}')
print(f'Coherence is None:    {result.coherence is None}')

## Plot individual channels

`sid.bode_plot` only shows the first channel, so we plot both manually.

In [ ]:
w = result.frequency
G1_true = 1.0 / (1.0 - 0.5 * np.exp(-1j * w))
G2_true = 0.3 / (1.0 - 0.7 * np.exp(-1j * w))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

ax1.semilogx(w, 20 * np.log10(np.abs(result.response[:, 0, 0])),
             'b', label='Estimated')
ax1.semilogx(w, 20 * np.log10(np.abs(G1_true)),
             'k--', label='True')
ax1.set_ylabel('Magnitude (dB)')
ax1.set_title(r'Channel 1: $G_1(z) = 1/(1 - 0.5z^{-1})$')
ax1.legend()
ax1.grid(True)

ax2.semilogx(w, 20 * np.log10(np.abs(result.response[:, 1, 0])),
             'r', label='Estimated')
ax2.semilogx(w, 20 * np.log10(np.abs(G2_true)),
             'k--', label='True')
ax2.set_ylabel('Magnitude (dB)')
ax2.set_xlabel('Frequency (rad/sample)')
ax2.set_title(r'Channel 2: $G_2(z) = 0.3/(1 - 0.7z^{-1})$')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## Noise spectral matrix

For a 2-output system, `noise_spectrum` is `(nf, 2, 2)`: a Hermitian
positive semi-definite matrix at each frequency. The diagonal elements
are the individual output noise spectra.

In [ ]:
diag11 = np.real(result.noise_spectrum[:, 0, 0])
diag22 = np.real(result.noise_spectrum[:, 1, 1])

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(w, 10 * np.log10(diag11), 'b', label=r'$\Phi_{v,11}$')
ax.semilogx(w, 10 * np.log10(diag22), 'r', label=r'$\Phi_{v,22}$')
ax.set_xlabel('Frequency (rad/sample)')
ax.set_ylabel('Noise Spectrum (dB)')
ax.set_title('Diagonal Elements of Noise Spectral Matrix')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## 2-output, 2-input system

Full 2x2 transfer matrix: each output depends on both inputs.

- $y_1 = G_{11} u_1 + G_{12} u_2 + v_1$
- $y_2 = G_{21} u_1 + G_{22} u_2 + v_2$

In [ ]:
rng2 = np.random.default_rng(20)

N2 = 3000
u2in = rng2.standard_normal((N2, 2))

y2out_1 = (lfilter([1],   [1, -0.5], u2in[:, 0])
         + lfilter([0.3], [1, -0.3], u2in[:, 1])
         + 0.1 * rng2.standard_normal(N2))
y2out_2 = (lfilter([0.5], [1, -0.7], u2in[:, 0])
         + lfilter([1],   [1, -0.4], u2in[:, 1])
         + 0.1 * rng2.standard_normal(N2))
y2out = np.column_stack([y2out_1, y2out_2])

result_22 = sid.freq_bt(y2out, u2in, window_size=30)

# Response is now (nf, 2, 2)
print(f'2x2 MIMO Response shape: {result_22.response.shape}')

In [ ]:
# Plot the full transfer matrix
titles = [['$G_{11}$', '$G_{12}$'],
          ['$G_{21}$', '$G_{22}$']]

fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True)
for iy in range(2):
    for iu in range(2):
        ax = axes[iy, iu]
        ax.semilogx(w, 20 * np.log10(np.abs(result_22.response[:, iy, iu])), 'b')
        ax.set_ylabel('Magnitude (dB)')
        if iy == 1:
            ax.set_xlabel('Frequency (rad/sample)')
        ax.set_title(titles[iy][iu])
        ax.grid(True)

plt.tight_layout()
plt.show()

## MIMO uncertainty

In v1.0, `response_std` is NaN for MIMO (no asymptotic formula implemented).

In [ ]:
print(f'MIMO response_std contains NaN: {np.all(np.isnan(result_22.response_std))}')